# v137b — All-Data 強検出器 & 1対1 連続融合 + ピクセル多数決 最終提出 CSV 生成・厳密検証ノートブック

180 枚の公式テスト画像 (`2048x2048`) に対し、`prod` と一字一句同一の前処理 (`solar_three_channel` JPEG q96)・Refiner (`mediseg_edgeattnet_refiner_all_data_resume.pt`)・排他化 (`drop_contained@0.90` + 面積昇順 ownership + `min_area=20`) を適用し、以下を一括生成・厳密検証します：
1. `repro_v96full` (`yolo_full @ 0.26` 再現サニティチェック)
2. `v133c` 単体 (`Small-TAL` All-Data 70ep)
3. **471枚 2-Fold 盲検 100% 勝者**: `v137_cons2of3_trio_b_c26_maj3` (`prod + yolo26 + v133c Small-TAL`, 2-of-3 @ `0.26` + `majority_3plus`、`+0.810 pt` PQ / `P(>0)=100%`)
4. **Fold-2 直交 3 検出器勝者**: `v137_cons2of3_sptal_prod_z_c26_maj3` (`prod + yolo26 + v134c Spine+Small-TAL`、`v134c` が終盤進行中の場合は自動待機して一括生成、`+1.380 pt` PQ / Sparse PQ `0.4028`)

全出力 CSV は保存前に **`validate_submission`（2048×2048 RLE 復号・重複 ID ゼロ・ピクセル重複ゼロ検査）** と **`submission_prod.csv` との 1対1 Hungarian IoU 照合** を通過させます。


In [ ]:
# [1/4] 環境・Drive 強制再マウント・All-Data 重み診断 & v136 勝者設定の読込
%pip install -q "ultralytics==8.4.120" pycocotools scipy
import hashlib, json, math, os, pickle, shutil, tempfile, time
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Iterable

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
from pycocotools import mask as mask_utils
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm
from ultralytics import YOLO

drive.mount("/content/drive", force_remount=True)
assert torch.cuda.is_available(), "GPU ランタイムを使用してください"

DRIVE_ROOT = Path("/content/drive/MyDrive/solar_filament_v30")
ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
OUT_DIR = ARTIFACT_ROOT / "submissions_v137"
OUT_DIR.mkdir(parents=True, exist_ok=True)
TEST_CACHE_DIR = OUT_DIR / "test_det_caches"
TEST_CACHE_DIR.mkdir(parents=True, exist_ok=True)

TEST_RAW_DIR = DRIVE_ROOT / "competition_data" / "MAGFiLO_1.0_Kaggle_2026" / "test" / "test_images"
LOCAL_TEST_IMG = Path("/content/solar_v137/test_preproc")
LOCAL_WEIGHTS = Path("/content/solar_v137/weights")
LOCAL_TEST_IMG.mkdir(parents=True, exist_ok=True)
LOCAL_WEIGHTS.mkdir(parents=True, exist_ok=True)

test_files = sorted(p.name for p in TEST_RAW_DIR.glob("*.jpeg"))
assert len(test_files) == 180, f"Expected 180 test images, got {len(test_files)}"

REFINER_ALL = ARTIFACT_ROOT / "mediseg_edgeattnet_refiner_all_data_resume.pt"
assert REFINER_ALL.exists(), f"Missing all-data refiner: {REFINER_ALL}"

INCLUDE_LAST_CKPTS = False

RUN_DIRS = {
    "prod": DRIVE_ROOT / "runs" / "v96_data_scaling" / "yolo_full_fold2",
    "yolo26": DRIVE_ROOT / "runs" / "v123_yolo26_all" / "yolo_yolo26m_all_fold0",
    "v133c": DRIVE_ROOT / "runs" / "v133_small_tal" / "yolo_small_tal_all_fold2",
    "v134c": DRIVE_ROOT / "runs" / "v134_spine_small_tal" / "yolo_spine_small_tal_all_fold2",
    "v138c": DRIVE_ROOT / "runs" / "v138_spine_aux" / "yolo_spine_aux_all_fold2",
}

CANDIDATE_PTS = {
    "prod": ("prod", RUN_DIRS["prod"] / "weights" / "best.pt"),
    "yolo26": ("yolo26", RUN_DIRS["yolo26"] / "weights" / "best.pt"),
    "tal_best": ("v133c", RUN_DIRS["v133c"] / "weights" / "best.pt"),
    "tal_last": ("v133c", RUN_DIRS["v133c"] / "weights" / "last.pt"),
    "sptal_best": ("v134c", RUN_DIRS["v134c"] / "weights" / "best.pt"),
    "sptal_last": ("v134c", RUN_DIRS["v134c"] / "weights" / "last.pt"),
    "sp_best": ("v138c", RUN_DIRS["v138c"] / "weights" / "best.pt"),
    "sp_last": ("v138c", RUN_DIRS["v138c"] / "weights" / "last.pt"),
}

def inspect_run_dir(run_key: str, run_dir: Path, target_epochs: int = 70):
    # Force FUSE directory cache refresh on Google Drive
    for d in [DRIVE_ROOT / "runs", run_dir.parent, run_dir, run_dir / "weights"]:
        try:
            if d.exists():
                os.listdir(str(d))
        except OSError:
            pass
    if not run_dir.exists():
        return {"exists": False, "complete": False, "epochs": 0, "best_mb": 0.0, "tc": False, "last_age_s": 1e9}
    tc = (run_dir / "TRAINING_COMPLETE").exists()
    rc = run_dir / "results.csv"
    epochs = 0
    if rc.exists():
        try:
            lines = [ln for ln in rc.read_text(encoding="utf-8", errors="ignore").splitlines() if ln.strip()]
            epochs = max(0, len(lines) - 1)
        except OSError:
            pass
    bp = run_dir / "weights" / "best.pt"
    lp = run_dir / "weights" / "last.pt"
    best_mb = (bp.stat().st_size / (1024 * 1024)) if bp.exists() else 0.0
    last_age_s = (time.time() - lp.stat().st_mtime) if lp.exists() else 1e9
    if run_key in ("prod", "yolo26"):
        complete = bp.exists() and best_mb > 30.0
    else:
        # Completed Ultralytics 70ep run has TRAINING_COMPLETE or stripped best.pt (30-100 MB) after >=70 epochs
        complete = bp.exists() and (tc or (epochs >= target_epochs and 30.0 < best_mb < 100.0))
    return {
        "exists": True,
        "complete": complete,
        "epochs": epochs,
        "best_mb": round(best_mb, 2),
        "tc": tc,
        "last_age_s": round(last_age_s, 1),
    }

def refresh_available_pts(include_last: bool = False, verbose: bool = True):
    statuses = {rk: inspect_run_dir(rk, rd) for rk, rd in RUN_DIRS.items()}
    if verbose:
        print("=== All-Data Run 状態診断 ===")
        for rk, st in statuses.items():
            print(f"  {rk:8s} | exists={str(st['exists']):5s} | complete={str(st['complete']):5s} | tc={str(st['tc']):5s} | epochs={st['epochs']:2d} | best_mb={st['best_mb']:6.2f}")
    avail = {}
    for k, (rk, p) in CANDIDATE_PTS.items():
        if k.endswith("_last") and not include_last:
            continue
        if statuses[rk]["complete"] and p.exists():
            avail[k] = p
    return avail, statuses

AVAILABLE_PTS, RUN_STATUSES = refresh_available_pts(include_last=INCLUDE_LAST_CKPTS, verbose=True)

assert "prod" in AVAILABLE_PTS and "yolo26" in AVAILABLE_PTS and "tal_best" in AVAILABLE_PTS, (
    f"v133c (All-Data Small-TAL) が見つかりません。診断結果: {RUN_STATUSES}"
)
print("利用可能な All-Data 検出器:", {k: str(v) for k, v in AVAILABLE_PTS.items()})


In [ ]:
# [2/4] 前処理・Refiner・1対1融合・2048x2048 厳密検証器の定義
"""
MediSeg-AI Knowledge Port for Solar Filament Segmentation
Includes:
1. MediSeg UNet
2. MediSeg DeepLabV3Plus with ASPP (Atrous Spatial Pyramid Pooling)
3. CombinedLoss (BCE + Dice Loss)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional

# ============================================================
# 1. Combined Loss (BCE + Dice Loss)
# ============================================================
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)
        
        intersection = (probs * targets).sum()
        dice = (2.0 * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1.0 - dice

class CombinedLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        dice_loss = self.dice(logits, targets)
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

# ============================================================
# 2. MediSeg UNet Architecture
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, mid_channels: Optional[int] = None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, bilinear: bool = True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class MediSegUNet(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1, bilinear: bool = False):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

# ============================================================
# 3. MediSeg DeepLabV3Plus with ASPP Module
# ============================================================
class ASPP(nn.Module):
    def __init__(self, in_channels: int, out_channels: int = 256):
        super().__init__()
        self.conv1x1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1x1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=6, dilation=6, bias=False)
        self.bn3x3_1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_2 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=12, dilation=12, bias=False)
        self.bn3x3_2 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_3 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=18, dilation=18, bias=False)
        self.bn3x3_3 = nn.BatchNorm2d(out_channels)
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv_global = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn_global = nn.BatchNorm2d(out_channels)
        
        self.conv_final = nn.Conv2d(out_channels * 5, out_channels, kernel_size=1, bias=False)
        self.bn_final = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        size = x.shape[2:]
        x1 = self.relu(self.bn1x1(self.conv1x1(x)))
        x2 = self.relu(self.bn3x3_1(self.conv3x3_1(x)))
        x3 = self.relu(self.bn3x3_2(self.conv3x3_2(x)))
        x4 = self.relu(self.bn3x3_3(self.conv3x3_3(x)))
        
        x5 = self.global_avg_pool(x)
        x5 = self.relu(self.bn_global(self.conv_global(x5)))
        x5 = F.interpolate(x5, size=size, mode='bilinear', align_corners=True)
        
        x_cat = torch.cat([x1, x2, x3, x4, x5], dim=1)
        return self.relu(self.bn_final(self.conv_final(x_cat)))

class MediSegDeepLabV3Plus(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1):
        super().__init__()
        # Feature Extractor
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 512, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )
        self.aspp = ASPP(512, 256)
        self.classifier = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, out_channels, kernel_size=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_size = x.shape[2:]
        feat = self.stem(x)
        aspp_feat = self.aspp(feat)
        out = self.classifier(aspp_feat)
        return F.interpolate(out, size=input_size, mode='bilinear', align_corners=True)

# ============================================================
# 4. Edge-Guided Attention U-Net (EG-MHSA)
# ============================================================
# 2026-08-29、WINNING_STRATEGY_2026-08-29.md §1.4のPhase 2実験用に追加。
# 参考文献: "EdgeAttNet: Towards Barb-Aware Filament Segmentation"
# (arXiv:2509.02964)。MAGFiLO上でU-Net(mIoU_pairwise 0.5724)から
# EdgeAttNet(0.6451)への改善を報告しており、barb(フィラメントの微細な
# 枝構造)の認識精度向上を狙ったアーキテクチャ。本プロジェクトの
# refiner(MediSegUNet)が抱える主要な弱点(near-miss境界品質)と直接
# 対応するため、置き換え候補として追試する。
#
# 【重要な注意】著者の公開リポジトリ(github.com/dasjar/EdgeAttNet)を
# 直接確認したところ、`model/edgeattnet_model.py`(本来UNetEdgeTransformer
# クラスが定義されているべきファイル)の中身が誤って`main.py`と同一に
# なっており、実際のアーキテクチャコードはリポジトリ上に存在しない
# (2026-08-29、`gh api`でリポジトリ全ファイルを走査して確認)。学習済み
# 重みの配布も確認できなかった。**以下は論文本文のアーキテクチャ記述
# (edge prior E=σ(Conv3x3(入力画像))をbottleneckへbilinear補間し
# 1x1convで射影、Q・Kへ加算、位置エンコーディングなし)のみを根拠にした
# 独自の再実装であり、著者オリジナル実装とのコード上の一致は未検証。**
# ベースにするU-Net本体はMediSegUNetをそのまま流用しているため、
# パラメータ総数は論文記載の値(22.6M)とは一致しない。


class EdgeGuidedMHSA(nn.Module):
    """Edge-Guided Multi-Head Self-Attention (EG-MHSA)。

    U-NetのbottleneckでSelf-Attentionを行う際、入力画像から抽出した
    エッジ事前分布をQuery/Keyに加算し、境界・barb構造への感度を高める。
    Value・出力チャネル数はbottleneckの特徴マップと同一に保つ
    (MediSegUNetへdrop-inできるよう設計)。
    """

    def __init__(self, in_channels: int, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        # E = sigmoid(Conv3x3(入力画像)): 単一チャンネルのエッジ事前分布
        self.edge_conv = nn.Conv2d(in_channels, 1, kernel_size=3, padding=1)
        # E' = Conv1x1(interpolate(E)): bottleneckのチャンネル数へ射影
        self.edge_proj = nn.Conv2d(1, bottleneck_channels, kernel_size=1)
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor, original_image: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        edge_map = torch.sigmoid(self.edge_conv(original_image))
        edge_map = F.interpolate(edge_map, size=(h, w), mode="bilinear", align_corners=True)
        edge_prior = self.edge_proj(edge_map)  # (B, C, H, W)

        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        edge_flat = edge_prior.flatten(2).transpose(1, 2)  # (B, N, C)
        qk = x_flat + edge_flat  # Q = K = x + E'。位置エンコーディングは使わない(論文通り)

        attn_out, _ = self.mhsa(qk, qk, x_flat)  # V = x(エッジ由来の加算なし)
        attn_out = self.norm(attn_out + x_flat)  # 残差接続 + LayerNorm(論文に明記はないが標準的なTransformerブロックとして採用)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegEdgeAttUNet(nn.Module):
    """MediSegUNetのbottleneckにEdgeGuidedMHSAを挿入したdrop-in代替。

    入出力インターフェースはMediSegUNetと完全に同一(in_channels=3の
    crop、out_channels=1の確率マップ)。既存のrefiner学習・推論コードを
    変更せず、モデルクラスの差し替えだけで比較できるように設計している。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.edge_attn = EdgeGuidedMHSA(in_channels, bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.edge_attn(x5, x)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)


# ============================================================
# 5. Plain MHSA U-Net (matched-parameter ablation control, no edge prior)
# ============================================================
# 2026-08-31、report/main.tex Future Work item 1(matched-parameter
# ablation)のために追加。EdgeGuidedMHSAからedge_conv/edge_projを除いた
# だけの構成——Q=K=V=bottleneck特徴量の標準的なself-attentionで、
# edge priorをQ/Kへ加算する項だけがない。edge_conv/edge_projは
# 3(in_channels)*1*3*3+1 + 1*1024*1*1+1024 ≈ 2,076パラメータしかなく
# 全体(約35.2M)の0.006%程度なので、このクラスはMediSegEdgeAttUNetと
# ほぼ完全にパラメータ数が一致する("U-Net + MHSA, no positional
# encoding"に相当、EdgeAttNet論文のablationと同じ切り分け)。
# report/main.texが指摘してきた「edge priorそのものの効果か、単に
# attention容量が増えただけか」という交絡を分離するための唯一の差分。


class PlainMHSA(nn.Module):
    """edge priorなしの標準Multi-Head Self-Attention(bottleneck上)。

    EdgeGuidedMHSAと同じchannel数・head数・残差接続+LayerNormを保ちつつ、
    Q・Kへの edge_prior 加算だけを行わない(Q=K=V=bottleneck特徴量)。
    """

    def __init__(self, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        attn_out, _ = self.mhsa(x_flat, x_flat, x_flat)  # Q=K=V=x(edge priorなし)
        attn_out = self.norm(attn_out + x_flat)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegPlainMHSAUNet(nn.Module):
    """MediSegEdgeAttUNetからedge priorだけを除いた matched-parameter 対照群。

    入出力インターフェースはMediSegUNet/MediSegEdgeAttUNetと完全に同一。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.plain_attn = PlainMHSA(bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.plain_attn(x5)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)

"""1-to-1 Multi-detector bipartite matching and continuous score/mask fusion (Idea 8, v132).

Fixes three structural issues in ``run_fold_oof_eval_consensus.py`` (v120/v121c/v124c):
1. Replaces unconstrained transitive union-find with greedy descending-IoU 1-to-1 bipartite
   clustering (at most 1 proposal per detector in every cluster), preventing adjacent-filament
   chaining across detectors.
2. Replaces hard step-function vote counting (``count(s >= 0.26) >= 2``, which discards
   ``(0.45, 0.24, 0)``, ``(0.24, 0.24, 0.24)``, and solo ``(0.75, 0, 0)``) with continuous
   calibrated cluster scoring ``sum_d w_d * s_d`` (missing detector contributes ``0.0``).
3. Supports score-weighted multi-detector proposal mask fusion before ``MediSegEdgeAttUNet``.
"""


from dataclasses import dataclass

import numpy as np
from pycocotools import mask as mask_utils


@dataclass(frozen=True)
class DetectorProposal:
    det_name: str
    conf: float
    raw_rle: dict
    prop_idx: int


@dataclass
class MatchedCluster:
    members: dict[str, DetectorProposal]  # det_name -> DetectorProposal (strictly <=1 per det)

    def score_of(self, det_name: str) -> float:
        p = self.members.get(det_name)
        return float(p.conf) if p is not None else 0.0

    def weighted_score(self, weights: dict[str, float]) -> float:
        return float(sum(w * self.score_of(d) for d, w in weights.items()))

    def max_conf(self) -> float:
        return max((float(p.conf) for p in self.members.values()), default=0.0)

    def count_above(self, thr: float) -> int:
        return sum(1 for p in self.members.values() if float(p.conf) >= thr)

    def best_member(self) -> DetectorProposal:
        return max(self.members.values(), key=lambda p: (float(p.conf), p.det_name))


def match_detector_proposals_1to1(
    proposals: list[DetectorProposal],
    *,
    iou_threshold: float = 0.45,
) -> list[MatchedCluster]:
    """Group cross-detector proposals by descending mask IoU with <=1 member per detector."""
    n = len(proposals)
    if n == 0:
        return []
    rles = [p.raw_rle for p in proposals]
    iou_mat = np.asarray(mask_utils.iou(rles, rles, [0] * n), dtype=np.float64)

    parent = list(range(n))
    det_sets = [{proposals[i].det_name} for i in range(n)]

    def find(x: int) -> int:
        root = x
        while parent[root] != root:
            root = parent[root]
        while parent[x] != root:
            parent[x], x = root, parent[x]
        return root

    pairs: list[tuple[float, int, int]] = []
    for i in range(n):
        di = proposals[i].det_name
        for j in range(i + 1, n):
            if di == proposals[j].det_name:
                continue
            v = float(iou_mat[i, j])
            if v >= iou_threshold:
                pairs.append((v, i, j))
    pairs.sort(key=lambda t: (-t[0], t[1], t[2]))

    for _, i, j in pairs:
        ri, rj = find(i), find(j)
        if ri != rj and det_sets[ri].isdisjoint(det_sets[rj]):
            parent[rj] = ri
            det_sets[ri] |= det_sets[rj]

    grouped: dict[int, dict[str, DetectorProposal]] = {}
    for idx in range(n):
        r = find(idx)
        p = proposals[idx]
        grouped.setdefault(r, {})[p.det_name] = p

    clusters = [MatchedCluster(members=m) for m in grouped.values()]
    clusters.sort(key=lambda c: -c.max_conf())
    return clusters


def fuse_cluster_proposal_mask(
    cluster: MatchedCluster,
    *,
    mode: str = "max_conf",
    vote_threshold: float = 0.45,
) -> np.ndarray:
    """Build bool proposal mask for ``MediSegEdgeAttUNet`` from a matched cluster."""
    if mode == "max_conf" or len(cluster.members) == 1:
        best = cluster.best_member()
        return mask_utils.decode(best.raw_rle) != 0
    if mode != "soft_vote":
        raise ValueError(f"Unknown mask fusion mode: {mode}")

    acc: np.ndarray | None = None
    w_sum = 0.0
    for p in cluster.members.values():
        w = max(float(p.conf), 1e-4)
        m = (mask_utils.decode(p.raw_rle) != 0).astype(np.float32)
        acc = (w * m) if acc is None else (acc + w * m)
        w_sum += w
    assert acc is not None
    return (acc / max(w_sum, 1e-6)) >= float(vote_threshold)


CropSpec = tuple[int, int, int, int, np.ndarray, float, tuple[int, int]]


def rle_to_crop_spec(rle: dict) -> CropSpec:
    """Decode a mask RLE once into a compact bounding-box crop tuple (~6 KB vs 4.2 MB)."""
    m = mask_utils.decode(rle) != 0
    h, w = m.shape
    x, y, bw, bh = [round(float(v)) for v in mask_utils.toBbox(rle)]
    if bw <= 0 or bh <= 0:
        return (0, 0, 0, 0, np.zeros((0, 0), dtype=bool), 0.0, (h, w))
    x0, y0 = max(0, x), max(0, y)
    x1, y1 = min(w, x + bw), min(h, y + bh)
    sub = np.ascontiguousarray(m[y0:y1, x0:x1])
    return (y0, y1, x0, x1, sub, float(sub.sum()), (h, w))


def fuse_refined_crop_specs(
    cluster: MatchedCluster,
    specs_by_det: dict[str, list[CropSpec]],
    *,
    mode: str = "best_member",
) -> CropSpec:
    """Fuse already-refined ``CropSpec`` masks across members of a 1-to-1 matched cluster.

    Modes:
    - ``"best_member"``: return the refined ``CropSpec`` of ``cluster.best_member()``.
    - ``"majority_3plus"``: when ``>=3`` non-empty members agree on an instance, take pixelwise
      majority across their refined crops (unweighted ``>=(K+1)//2`` for odd ``K``, confidence-
      weighted ``>=0.5*sum(w)`` for even ``K`` to break 2-vs-2 ties cleanly); fall back to
      ``best_member`` when ``<=2`` members match so thin filament tips are never clipped by
      2-way strict intersection.
    - ``"soft_w50"``: confidence-weighted pixel average ``>= 0.5 * sum(w)`` for ``>=2`` members.
    """
    bm = cluster.best_member()
    bm_spec = specs_by_det[bm.det_name][bm.prop_idx]
    if mode == "best_member" or len(cluster.members) == 1:
        return bm_spec

    member_specs: list[CropSpec] = []
    weights: list[float] = []
    for dname, prop in cluster.members.items():
        sp = specs_by_det[dname][prop.prop_idx]
        if sp[5] > 0.0:
            member_specs.append(sp)
            weights.append(max(float(prop.conf), 1e-4))
    k = len(member_specs)
    if k == 0:
        return bm_spec
    if k == 1 or (mode == "majority_3plus" and k < 3):
        return bm_spec if bm_spec[5] > 0.0 else member_specs[0]

    hw = member_specs[0][6]
    y0 = min(sp[0] for sp in member_specs)
    y1 = max(sp[1] for sp in member_specs)
    x0 = min(sp[2] for sp in member_specs)
    x1 = max(sp[3] for sp in member_specs)
    acc = np.zeros((y1 - y0, x1 - x0), dtype=np.float32)

    if mode == "majority_3plus" and (k % 2 == 1):
        for sy0, sy1, sx0, sx1, sub, _, _ in member_specs:
            acc[sy0 - y0 : sy1 - y0, sx0 - x0 : sx1 - x0] += sub.astype(np.float32)
        fused = acc >= float((k + 1) // 2)
    elif mode in ("majority_3plus", "soft_w50"):
        w_sum = float(sum(weights))
        for (sy0, sy1, sx0, sx1, sub, _, _), w in zip(member_specs, weights):
            acc[sy0 - y0 : sy1 - y0, sx0 - x0 : sx1 - x0] += w * sub.astype(np.float32)
        fused = acc >= (0.50 * w_sum)
    else:
        raise ValueError(f"Unknown refined crop fusion mode: {mode}")

    ys, xs = np.nonzero(fused)
    if len(xs) == 0:
        return (0, 0, 0, 0, np.zeros((0, 0), dtype=bool), 0.0, hw)
    ny0, ny1 = int(ys.min()), int(ys.max()) + 1
    nx0, nx1 = int(xs.min()), int(xs.max()) + 1
    sub_f = np.ascontiguousarray(fused[ny0:ny1, nx0:nx1])
    return (y0 + ny0, y0 + ny1, x0 + nx0, x0 + nx1, sub_f, float(len(xs)), hw)


def fast_apply_own_specs(
    specs: list[CropSpec],
    thr: float = 0.90,
    min_area: int = 20,
) -> list[dict]:
    """BBox-cropped ownership matching ``ownership.apply_ownership`` (drop_contained_then_area_asc)."""
    n = len(specs)
    if n == 0:
        return []
    h, w = specs[0][6]
    dropped: set[int] = set()
    for i in range(n):
        for j in range(i + 1, n):
            if i in dropped or j in dropped:
                continue
            y0i, y1i, x0i, x1i, sub_i, ai, _ = specs[i]
            y0j, y1j, x0j, x1j, sub_j, aj, _ = specs[j]
            if ai <= 0.0 and aj <= 0.0:
                continue
            smaller = i if ai <= aj else j
            smaller_area = min(ai, aj)
            iy0, iy1 = max(y0i, y0j), min(y1i, y1j)
            ix0, ix1 = max(x0i, x0j), min(x1i, x1j)
            if iy1 <= iy0 or ix1 <= ix0 or (iy1 - iy0) * (ix1 - ix0) < thr * max(smaller_area, 1.0):
                continue
            inter = int(
                np.logical_and(
                    sub_i[iy0 - y0i : iy1 - y0i, ix0 - x0i : ix1 - x0i],
                    sub_j[iy0 - y0j : iy1 - y0j, ix0 - x0j : ix1 - x0j],
                ).sum()
            )
            if inter / max(smaller_area, 1.0) >= thr:
                dropped.add(smaller)

    keep = [i for i in range(n) if i not in dropped]
    if not keep:
        return []
    occ = np.zeros((h, w), dtype=bool)
    kept_subs: list[tuple[int, int, int, int, np.ndarray]] = []
    for idx in sorted(keep, key=lambda i: specs[i][5]):
        y0, y1, x0, x1, sub, _, _ = specs[idx]
        if sub.size == 0:
            continue
        ex = sub & ~occ[y0:y1, x0:x1]
        if int(ex.sum()) >= min_area:
            occ[y0:y1, x0:x1] |= ex
            kept_subs.append((y0, y1, x0, x1, ex))
    if not kept_subs:
        return []
    canvas = np.zeros((h, w, len(kept_subs)), dtype=np.uint8, order="F")
    for k, (y0, y1, x0, x1, ex) in enumerate(kept_subs):
        canvas[y0:y1, x0:x1, k] = ex
    return mask_utils.encode(canvas)

"""V32 overlap-aware repair for Solar Filament Kaggle submissions."""


from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Iterable

import numpy as np
import pandas as pd


HEIGHT = 2048
WIDTH = 2048


@dataclass(frozen=True)
class MergeConfig:
    name: str
    min_dice: float
    min_containment: float = 0.0
    min_area: int = 20


def decode_compressed_counts(text: str) -> list[int]:
    counts: list[int] = []
    position = 0
    while position < len(text):
        value = 0
        shift = 0
        while True:
            code = ord(text[position]) - 48
            position += 1
            value |= (code & 0x1F) << (5 * shift)
            shift += 1
            if not code & 0x20:
                if code & 0x10:
                    value |= -1 << (5 * shift)
                break
        if len(counts) > 2:
            value += counts[-2]
        counts.append(value)
    return counts


def decode_mask(text: str, height: int = HEIGHT, width: int = WIDTH) -> np.ndarray:
    flat = np.zeros(height * width, dtype=bool)
    offset = 0
    foreground = False
    for run in decode_compressed_counts(text):
        if run < 0 or offset + run > flat.size:
            raise ValueError("Invalid compressed RLE run")
        if foreground and run:
            flat[offset : offset + run] = True
        offset += run
        foreground = not foreground
    if offset != flat.size:
        raise ValueError(f"RLE size mismatch: {offset} != {flat.size}")
    return flat.reshape((height, width), order="F")


def encode_compressed_counts(counts: Iterable[int]) -> str:
    counts = [int(value) for value in counts]
    output: list[str] = []
    for index, original in enumerate(counts):
        value = original - counts[index - 2] if index > 2 else original
        while True:
            code = value & 0x1F
            value >>= 5
            more = (code & 0x10 and value != -1) or (not code & 0x10 and value != 0)
            if more:
                code |= 0x20
            output.append(chr(code + 48))
            if not more:
                break
    return "".join(output)


def encode_mask(mask: np.ndarray) -> str:
    flat = np.asarray(mask, dtype=np.uint8).reshape(-1, order="F")
    padded = np.concatenate((np.zeros(1, dtype=np.uint8), flat, np.zeros(1, dtype=np.uint8)))
    changes = np.flatnonzero(np.diff(padded))
    boundaries = np.concatenate(([0], changes, [flat.size]))
    counts = np.diff(boundaries).astype(np.int64).tolist()
    if counts and counts[-1] == 0:
        counts.pop()
    return encode_compressed_counts(counts)


def overlap_metrics(first: np.ndarray, second: np.ndarray, first_area: int, second_area: int) -> tuple[float, float, int]:
    intersection = int(np.logical_and(first, second).sum())
    if intersection == 0:
        return 0.0, 0.0, 0
    containment = intersection / min(first_area, second_area)
    dice = 2.0 * intersection / (first_area + second_area)
    return containment, dice, intersection


def _components(size: int, edges: Iterable[tuple[int, int]]) -> list[list[int]]:
    parent = list(range(size))

    def find(index: int) -> int:
        while parent[index] != index:
            parent[index] = parent[parent[index]]
            index = parent[index]
        return index

    for first, second in edges:
        first_root, second_root = find(first), find(second)
        if first_root != second_root:
            parent[second_root] = first_root

    grouped: dict[int, list[int]] = defaultdict(list)
    for index in range(size):
        grouped[find(index)].append(index)
    return list(grouped.values())


def cluster_masks(
    masks: list[np.ndarray],
    config: MergeConfig,
) -> tuple[list[list[int]], list[tuple[int, int, float, float, int]]]:
    areas = [int(mask.sum()) for mask in masks]
    accepted: list[tuple[int, int]] = []
    audit: list[tuple[int, int, float, float, int]] = []
    for first in range(len(masks)):
        for second in range(first + 1, len(masks)):
            containment, dice, intersection = overlap_metrics(
                masks[first], masks[second], areas[first], areas[second]
            )
            if intersection == 0:
                continue
            audit.append((first, second, containment, dice, intersection))
            if containment >= config.min_containment and dice >= config.min_dice:
                accepted.append((first, second))
    return _components(len(masks), accepted), audit


def build_candidate(
    source: pd.DataFrame,
    config: MergeConfig,
    progress: Callable | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    grouped: dict[str, list[tuple[int, str]]] = defaultdict(list)
    for row in source.itertuples(index=False):
        image_id, suffix = str(row.filament_id).rsplit("_", 1)
        grouped[image_id].append((int(suffix), str(row.segmentation_rle)))

    iterator = grouped.items()
    if progress is not None:
        iterator = progress(iterator, total=len(grouped), desc=config.name)

    rows: list[dict[str, object]] = []
    audits: list[dict[str, object]] = []
    for image_id, records in iterator:
        records = sorted(records)
        masks = [decode_mask(counts) for _, counts in records]
        components, overlap_audit = cluster_masks(masks, config)
        clusters: list[tuple[int, np.ndarray, int]] = []
        for component in components:
            merged = np.logical_or.reduce([masks[index] for index in component])
            priority = min(records[index][0] for index in component)
            clusters.append((priority, merged, len(component)))

        occupied = np.zeros((HEIGHT, WIDTH), dtype=bool)
        kept: list[tuple[np.ndarray, int]] = []
        removed_pixels = 0
        dropped = 0
        for _, merged, component_size in sorted(clusters, key=lambda item: item[0]):
            removed_pixels += int(np.logical_and(merged, occupied).sum())
            exclusive = merged & ~occupied
            if exclusive.sum() < config.min_area:
                dropped += 1
                continue
            occupied |= exclusive
            kept.append((exclusive, component_size))

        for counter, (mask, _) in enumerate(kept, start=1):
            rows.append(
                {
                    "filament_id": f"{image_id}_{counter}",
                    "segmentation_rle": encode_mask(mask),
                }
            )
        audits.append(
            {
                "image_id": image_id,
                "source_masks": len(masks),
                "overlapping_pairs": len(overlap_audit),
                "clusters": len(components),
                "merged_groups": sum(len(component) > 1 for component in components),
                "submitted": len(kept),
                "removed_cross_cluster_pixels": removed_pixels,
                "dropped_masks": dropped,
            }
        )

    candidate = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
    return candidate, pd.DataFrame(audits)


def validate_submission(frame: pd.DataFrame) -> dict[str, object]:
    if frame.columns.tolist() != ["filament_id", "segmentation_rle"]:
        raise AssertionError(frame.columns.tolist())
    if frame.empty or frame.isna().any().any():
        raise AssertionError("Submission is empty or contains null values")
    if frame["filament_id"].duplicated().any():
        raise AssertionError("Duplicate filament_id")

    occupied_by_image: dict[str, np.ndarray] = {}
    areas: list[int] = []
    for row in frame.itertuples(index=False):
        mask = decode_mask(str(row.segmentation_rle))
        if not mask.any():
            raise AssertionError(f"Empty mask: {row.filament_id}")
        image_id = str(row.filament_id).rsplit("_", 1)[0]
        occupied = occupied_by_image.setdefault(image_id, np.zeros((HEIGHT, WIDTH), dtype=bool))
        if np.logical_and(occupied, mask).any():
            raise AssertionError(f"Overlapping masks: {image_id}")
        occupied |= mask
        areas.append(int(mask.sum()))

    return {
        "rows": len(frame),
        "images": len(occupied_by_image),
        "overlap_pixels": 0,
        "area_min": min(areas),
        "area_median": int(np.median(areas)),
        "area_max": max(areas),
    }


def assert_codec_roundtrip(samples: Iterable[str]) -> None:
    for text in samples:
        decoded = decode_mask(text)
        encoded = encode_mask(decoded)
        if not np.array_equal(decoded, decode_mask(encoded)):
            raise AssertionError("Compressed RLE round-trip failed")


def load_submission(path: str | Path) -> pd.DataFrame:
    frame = pd.read_csv(path, keep_default_na=False)
    if frame.columns.tolist() != ["filament_id", "segmentation_rle"]:
        raise ValueError(f"Unexpected columns: {frame.columns.tolist()}")
    return frame

def robust_uint8(array, mask=None, low=0.5, high=99.5):
    values = array[mask] if mask is not None and mask.any() else array.reshape(-1)
    lo, hi = np.percentile(values, [low, high])
    return np.clip((array - lo) * 255.0 / max(hi - lo, 1e-6), 0, 255).astype(np.uint8)

def solar_three_channel(gray):
    gray = gray.astype(np.float32)
    disk = gray > max(4.0, np.percentile(gray, 8))
    background = cv2.GaussianBlur(gray, (0, 0), sigmaX=80, sigmaY=80)
    flattened = gray / np.maximum(background, 1.0)
    flat_u8 = robust_uint8(flattened, disk, 1.0, 99.0)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(16, 16)).apply(flat_u8)
    raw_u8 = robust_uint8(gray, disk, 0.5, 99.5)
    dark_detail = cv2.morphologyEx(clahe, cv2.MORPH_BLACKHAT, np.ones((31, 31), np.uint8))
    dark_detail = robust_uint8(dark_detail, disk, 0.0, 99.5)
    out = np.dstack([raw_u8, clahe, dark_detail])
    out[~disk] = 0
    return out


In [ ]:
# [3/4] Test 180枚の前処理ステージング & 各 All-Data 検出器 + EdgeAtt Refiner キャッシュ
def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

for name in tqdm(test_files, desc="Test 180枚 solar_three_channel 前処理"):
    dst = LOCAL_TEST_IMG / name
    if dst.exists():
        continue
    gray = None
    for attempt in range(4):
        try:
            raw = np.frombuffer((TEST_RAW_DIR / name).read_bytes(), np.uint8)
            gray = cv2.imdecode(raw, cv2.IMREAD_GRAYSCALE)
            if gray is not None:
                break
        except OSError:
            pass
        time.sleep(2.0 * (attempt + 1))
    assert gray is not None, f"Failed to decode test image: {name}"
    cv2.imwrite(str(dst), solar_three_channel(gray), [cv2.IMWRITE_JPEG_QUALITY, 96])

local_ref_pt = LOCAL_WEIGHTS / "refiner_all.pt"
shutil.copy2(REFINER_ALL, local_ref_pt)
refiner = MediSegEdgeAttUNet(3, 1, False).cuda().eval()
_ck = torch.load(local_ref_pt, map_location="cuda", weights_only=False)
refiner.load_state_dict(_ck.get("best_model") or _ck["model"], strict=True)
del _ck

def sq_box(mask, context=1.65, minimum=96):
    h, w = mask.shape
    ys, xs = np.where(mask)
    if not len(xs):
        return 0, 0, w, h
    cx, cy = (xs.min() + xs.max() + 1) / 2.0, (ys.min() + ys.max() + 1) / 2.0
    side = min(max(int(math.ceil(max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * context)), minimum), min(h, w))
    x0, y0 = round(cx - side / 2.0), round(cy - side / 2.0)
    x1, y1 = x0 + side, y0 + side
    if x0 < 0: x1 -= x0; x0 = 0
    if y0 < 0: y1 -= y0; y0 = 0
    if x1 > w: x0 -= x1 - w; x1 = w
    if y1 > h: y0 -= y1 - h; y1 = h
    return max(0, x0), max(0, y0), min(w, x1), min(h, y1)

def refine_batch(img_bgr, masks):
    if not masks:
        return []
    boxes = [sq_box(m) for m in masks]
    crops = np.stack([
        cv2.resize(img_bgr[y0:y1, x0:x1], (320, 320), interpolation=cv2.INTER_LINEAR).transpose(2, 0, 1)
        for x0, y0, x1, y1 in boxes
    ], 0)
    t = torch.from_numpy(crops).float().cuda() / 255.0
    outs = []
    with torch.no_grad():
        for i in range(0, len(t), 32):
            outs.append(torch.sigmoid(refiner(t[i:i + 32])).squeeze(1).cpu().numpy())
    probs = np.concatenate(outs, 0)
    res = []
    for m, (x0, y0, x1, y1), pr in zip(masks, boxes, probs):
        pb = cv2.resize(pr, (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR) >= 0.45
        fm = np.zeros_like(m, bool)
        fm[y0:y1, x0:x1] = pb
        res.append(fm & m)
    return res

TEST_CACHES = {}
WEIGHT_SHAS = {"refiner_all": file_sha256(local_ref_pt)}

def ensure_test_cache(dkey: str, src_pt: Path):
    if dkey in TEST_CACHES:
        return
    sha = file_sha256(src_pt)
    WEIGHT_SHAS[dkey] = sha
    cpkl = TEST_CACHE_DIR / f"test_det_{sha[:16]}.pkl"
    if cpkl.exists():
        TEST_CACHES[dkey] = pickle.loads(cpkl.read_bytes())
        print(f"再利用 [{dkey}]: {cpkl.name} (sha={sha[:12]})")
        return
    lpt = LOCAL_WEIGHTS / f"{dkey}.pt"
    shutil.copy2(src_pt, lpt)
    model = YOLO(str(lpt))
    cdict = {}
    for name in tqdm(test_files, desc=f"test 推論+refine [{dkey}]"):
        ip = LOCAL_TEST_IMG / name
        r = model.predict(str(ip), imgsz=1280, conf=0.08, iou=0.55, agnostic_nms=True, retina_masks=True, verbose=False)[0]
        h, w = r.orig_shape
        assert (h, w) == (2048, 2048), f"Unexpected test shape: {(h, w)}"
        ms = []
        if r.masks is not None:
            for d in r.masks.data.detach().cpu().numpy():
                ms.append((cv2.resize(d, (w, h), interpolation=cv2.INTER_LINEAR) if d.shape != (h, w) else d) >= 0.5)
        cf = r.boxes.conf.detach().cpu().numpy().astype(np.float32) if r.boxes is not None else np.zeros(0, np.float32)
        vi = [i for i, x in enumerate(ms) if float(x.sum()) >= 20.0]
        vms = [ms[i] for i in vi]
        vcf = cf[vi] if vi else np.zeros(0, np.float32)
        areas = [float(x.sum()) for x in vms]
        im = cv2.imread(str(ip), cv2.IMREAD_COLOR)
        rms = refine_batch(im, vms)
        raw_rles = [mask_utils.encode(np.asfortranarray(x.astype(np.uint8))) for x in vms]
        ref_rles = [mask_utils.encode(np.asfortranarray(x.astype(np.uint8))) for x in rms]
        cdict[name] = (vcf, areas, raw_rles, ref_rles)
    del model
    torch.cuda.empty_cache()
    cpkl.write_bytes(pickle.dumps(cdict))
    TEST_CACHES[dkey] = cdict
    print(f"保存完了 [{dkey}]: {cpkl.name} (sha={sha[:12]})")

for dkey, src_pt in list(AVAILABLE_PTS.items()):
    ensure_test_cache(dkey, src_pt)

# prod / yolo26 / tal_best の推論中 (~15分) に v134c (Spine+Small-TAL All-Data) が完走間近・完走済みなら自動回収
st_v134 = inspect_run_dir("v134c", RUN_DIRS["v134c"])
if not st_v134["complete"] and st_v134["epochs"] >= 62 and st_v134["last_age_s"] < 900:
    print(f"[v134c 自動待機] Epoch {st_v134['epochs']}/70 進行中 — 完走まで最大 25 分待機して sptal_best も一括生成します...")
    for _ in range(50):
        time.sleep(30)
        st_v134 = inspect_run_dir("v134c", RUN_DIRS["v134c"])
        print(f"  [v134c 監視] epochs={st_v134['epochs']}/70 | tc={st_v134['tc']} | best_mb={st_v134['best_mb']} | complete={st_v134['complete']}")
        if st_v134["complete"] or st_v134["last_age_s"] >= 900:
            break

AVAILABLE_PTS, RUN_STATUSES = refresh_available_pts(include_last=INCLUDE_LAST_CKPTS, verbose=True)
for dkey, src_pt in list(AVAILABLE_PTS.items()):
    ensure_test_cache(dkey, src_pt)


In [ ]:
# [4/4] 候補 CSV 生成・validate_submission 厳密検査・既存提出 CSV との Hungarian IoU 照合
SPECS_BY_DET = {
    dkey: {name: [rle_to_crop_spec(r) for r in row[3]] for name, row in cdict.items()}
    for dkey, cdict in TEST_CACHES.items()
}

def compare_with_reference(cand_df, ref_csv_path):
    if not ref_csv_path.exists():
        return None
    ref_df = pd.read_csv(ref_csv_path, keep_default_na=False)
    sub_a = defaultdict(list)
    sub_b = defaultdict(list)
    for row in cand_df.itertuples():
        sub_a[row.filament_id.rsplit("_", 1)[0]].append({"size": [2048, 2048], "counts": row.segmentation_rle.encode("utf-8")})
    for row in ref_df.itertuples():
        sub_b[row.filament_id.rsplit("_", 1)[0]].append({"size": [2048, 2048], "counts": row.segmentation_rle.encode("utf-8")})
    stems = sorted(set(sub_a) | set(sub_b))
    matched = 0
    ious = []
    for stem in stems:
        ra, rb = sub_a.get(stem, []), sub_b.get(stem, [])
        if not ra or not rb:
            continue
        mat = np.asarray(mask_utils.iou(ra, rb, [0] * len(rb)), np.float64)
        ious.extend(mat.max(axis=1).tolist())
        hits = mat > 0.5
        ri, ci = linear_sum_assignment(-hits.astype(np.int8))
        matched += int(hits[ri, ci].sum())
    return {
        "ref_rows": len(ref_df),
        "cand_rows": len(cand_df),
        "matched_1to1_iou05": matched,
        "match_rate_vs_ref": round(matched / max(len(ref_df), 1), 4),
        "mean_best_iou": round(float(np.mean(ious)), 4) if ious else 0.0,
    }

SUBMISSION_SPECS = [
    ("repro_v96full", {"prod": 1.0}, 0.26, None, "best_member"),
    ("v137_small_tal_all_best_026", {"tal_best": 1.0}, 0.26, None, "best_member"),
    ("v137_cons2of3_trio_b_c26_maj3", {"prod": 1.0, "yolo26": 1.0, "tal_best": 1.0}, 0.26, 2, "majority_3plus"),
    ("v137_cons2of3_trio_b_c26_bm", {"prod": 1.0, "yolo26": 1.0, "tal_best": 1.0}, 0.26, 2, "best_member"),
    ("v137_trio_all_tal45_y30_z25_t23_maj3", {"tal_best": 0.45, "prod": 0.30, "yolo26": 0.25}, 0.23, None, "majority_3plus"),
]
if "tal_last" in TEST_CACHES:
    SUBMISSION_SPECS.extend([
        ("v137_small_tal_all_last_026", {"tal_last": 1.0}, 0.26, None, "best_member"),
        ("v137_cons2of3_trio_l_c26_maj3", {"prod": 1.0, "yolo26": 1.0, "tal_last": 1.0}, 0.26, 2, "majority_3plus"),
    ])
if "sptal_best" in TEST_CACHES:
    SUBMISSION_SPECS.extend([
        ("v137_spine_small_tal_all_best_026", {"sptal_best": 1.0}, 0.26, None, "best_member"),
        ("v137_cons2of3_sptal_prod_z_c26_maj3", {"sptal_best": 1.0, "prod": 1.0, "yolo26": 1.0}, 0.26, 2, "majority_3plus"),
        ("v137_cons2of4_quad_b_c26_maj3", {"sptal_best": 1.0, "tal_best": 1.0, "prod": 1.0, "yolo26": 1.0}, 0.26, 2, "majority_3plus"),
        ("v137_quad_all_best_t22_maj3", {"sptal_best": 0.35, "tal_best": 0.30, "prod": 0.20, "yolo26": 0.15}, 0.22, None, "majority_3plus"),
    ])
if "sptal_last" in TEST_CACHES:
    SUBMISSION_SPECS.extend([
        ("v137_spine_small_tal_all_last_026", {"sptal_last": 1.0}, 0.26, None, "best_member"),
        ("v137_quad_all_last_t22_maj3", {"sptal_last": 0.35, "tal_last": 0.30, "prod": 0.20, "yolo26": 0.15}, 0.22, None, "majority_3plus"),
    ])
if "sp_best" in TEST_CACHES:
    SUBMISSION_SPECS.extend([
        ("v137_spine_aux_all_best_026", {"sp_best": 1.0}, 0.26, None, "best_member"),
        ("v137_cons2of3_spb_prod_z_c26_maj3", {"sp_best": 1.0, "prod": 1.0, "yolo26": 1.0}, 0.26, 2, "majority_3plus"),
        ("v137_cons2of4_spb_talb_prod_z_c26_maj3", {"sp_best": 1.0, "tal_best": 1.0, "prod": 1.0, "yolo26": 1.0}, 0.26, 2, "majority_3plus"),
    ])
    if "tal_last" in TEST_CACHES:
        SUBMISSION_SPECS.extend([
            ("v137_cons2of4_spb_tall_prod_z_c26_maj3", {"sp_best": 1.0, "tal_last": 1.0, "prod": 1.0, "yolo26": 1.0}, 0.26, 2, "majority_3plus"),
            ("v137_quad_spb40_tall25_y20_z15_t23_maj3", {"sp_best": 0.40, "tal_last": 0.25, "prod": 0.20, "yolo26": 0.15}, 0.23, None, "majority_3plus"),
        ])

# v136 の 2-Fold 盲検勝者が存在し、全構成重みが揃っていれば自動追加
v136_mf = ARTIFACT_ROOT / "v136_strong_fusion_2fold_manifest.json"
if v136_mf.exists():
    v136_data = json.loads(v136_mf.read_text(encoding="utf-8"))
    seen_arms = set()
    for tag_sel, blk in [("v136_f2sel_pq", v136_data["cross_selection"]["by_pq"]["selected_on_fold2"]),
                         ("v136_f2sel_sparse", v136_data["cross_selection"]["by_sparse_r077"]["selected_on_fold2"])]:
        arm_id = blk["arm"]
        if arm_id in seen_arms:
            continue
        seen_arms.add(arm_id)
        arm_cfg = v136_data["results"][arm_id]
        if all(k in TEST_CACHES for k in arm_cfg["weights"]):
            SUBMISSION_SPECS.append((
                f"v137_{tag_sel}_{arm_id}",
                arm_cfg["weights"],
                arm_cfg["threshold"],
                arm_cfg["discrete_vote"],
                arm_cfg.get("mask_mode", "best_member"),
            ))

REF_PROD_CSV = ARTIFACT_ROOT / "repro_prod_colab" / "submission_prod.csv"
REF_CONS_CSV = ARTIFACT_ROOT / "consensus_v126" / "submission_consensus.csv"

audit_summary = {"weights_sha256": WEIGHT_SHAS, "candidates": {}}

for sub_name, wdict, thr, dvote, mmode in SUBMISSION_SPECS:
    det_tuple = tuple(sorted(wdict.keys()))
    rows = []
    empty_imgs = []
    for name in test_files:
        if len(det_tuple) == 1:
            d0 = det_tuple[0]
            cf, areas, _, _ = TEST_CACHES[d0][name]
            chosen_specs = [SPECS_BY_DET[d0][name][i] for i in range(len(cf)) if cf[i] >= thr and areas[i] >= 20.0]
        else:
            props = []
            for d in det_tuple:
                cf, areas, raw_rles, _ = TEST_CACHES[d][name]
                for i in range(len(cf)):
                    if areas[i] >= 20.0:
                        props.append(DetectorProposal(d, float(cf[i]), raw_rles[i], i))
            clusters = match_detector_proposals_1to1(props, iou_threshold=0.45)
            s_by_det = {d: SPECS_BY_DET[d][name] for d in det_tuple}
            chosen_specs = []
            for cl in clusters:
                ok = (cl.count_above(thr) >= dvote) if dvote is not None else (cl.weighted_score(wdict) >= thr)
                if ok:
                    sp = fuse_refined_crop_specs(cl, s_by_det, mode=mmode)
                    if sp[5] >= 20.0:
                        chosen_specs.append(sp)
        final_rles = fast_apply_own_specs(chosen_specs)
        if not final_rles:
            empty_imgs.append(name)
        stem = Path(name).stem
        for idx_m, rle in enumerate(final_rles, start=1):
            rows.append({
                "filament_id": f"{stem}_{idx_m}",
                "segmentation_rle": rle["counts"].decode("utf-8"),
            })

    df = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
    val_rep = validate_submission(df)
    assert_codec_roundtrip(df["segmentation_rle"].iloc[:25].tolist())

    out_csv = OUT_DIR / f"submission_{sub_name}.csv"
    df.to_csv(out_csv, index=False, lineterminator="\n")
    csv_sha = file_sha256(out_csv)

    cmp_prod = compare_with_reference(df, REF_PROD_CSV)
    cmp_cons = compare_with_reference(df, REF_CONS_CSV)
    audit_summary["candidates"][sub_name] = {
        "csv_path": str(out_csv),
        "sha256": csv_sha,
        "weights": wdict,
        "threshold": thr,
        "discrete_vote": dvote,
        "mask_mode": mmode,
        "validator": val_rep,
        "empty_test_images_count": len(empty_imgs),
        "empty_test_images": empty_imgs,
        "inst_per_180_img": round(len(df) / 180.0, 3),
        "vs_prod_csv": cmp_prod,
        "vs_consensus_v126_csv": cmp_cons,
    }
    print(
        f"{sub_name:38s} | rows={len(df):4d} ({len(df)/180:.2f}/img) | imgs={val_rep['images']}/180 "
        f"| overlap={val_rep['overlap_pixels']} | vs_prod_match={cmp_prod['match_rate_vs_ref'] if cmp_prod else 'N/A'}"
    )

audit_path = OUT_DIR / "v137_submission_audit_manifest.json"
audit_path.write_text(json.dumps(audit_summary, indent=2, ensure_ascii=False), encoding="utf-8")
print("\nSaved submission audit manifest:", audit_path)
